# Mentions in the first entry

A mention is a span in the cleaned entry text. Three classes matter:

- `PROP` is a proper name (`Jonathan Harker`, `Bistritz`).
- `NOM` is a title used as a noun phrase (`the Count`).
- `PRON` is a pronoun. Pronouns attach only inside a sentence that has exactly one compatible person, and only in a later step.

Detection mixes a small gazetteer with `en_core_web_sm`. The gazetteer forces `Castle Dracula` to a facility and `the Demeter` to a vehicle. Longer spans win. A proper span that swallowed a quote, a stopword, or more than five tokens is dropped, so `the Castle Dracula` does not hide the gazetteer hit `Castle Dracula`.

The next cell loads the small spaCy model and runs entry 1 only. The full book is `python scripts/build_knowledge_graph.py`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "kg").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
import spacy

from src.kg.entries import load_entries
from src.kg.mentions import extract_mentions

nlp = spacy.load("en_core_web_sm")
entries = load_entries(ROOT / "data" / "dracula_ascii_rag.json")
mentions = extract_mentions([entries[0]], nlp=nlp)
print(len(mentions), "mentions in entry 1")
for mention in mentions:
    if mention.mention_class == "PRON":
        continue
    print(f"{mention.start:5} {mention.mention_class:4} {mention.entity_type:4} {mention.surface}")


## Checks on entry 1

Entry 1 really does say Bistritz, Count Dracula, and Mina more than once. Castle Dracula appears once. The small model tags some of these spans as organisations. Type correction against the gold alias list happens in `apply_alias_types` during the full build. The facility type on Castle Dracula comes from the gazetteer, so it is already right here.


In [ ]:
castle = [m for m in mentions if m.surface == "Castle Dracula"]
count = [m for m in mentions if m.surface == "Count Dracula"]
print("Castle Dracula", [(m.entity_type, m.mention_class) for m in castle])
print("Count Dracula", [(m.entity_type, m.mention_class) for m in count])
assert len(castle) == 1 and castle[0].entity_type == "FAC"
assert any(m.entity_type == "PER" for m in count)
assert any(m.surface == "Bistritz" for m in mentions)
assert any(m.surface == "Mina" for m in mentions)
print("entry 1 checks passed")
